#  Artificial Neural Networks 

In [ ]:
# import warnings
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import time
import tensorflow as tf
import random
import os
from scipy.stats import shapiro
from geopandas import GeoDataFrame
from pyproj import Proj
from shapely.geometry import Point
from sklearn.neighbors import BallTree
#from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from scikeras.wrappers import KerasRegressor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasRegressor  # ✅ Corrected import

myProj = Proj("+proj=utm +zone=43 +north +datum=WGS84 +units=m +no_defs")   # Define Projection to be UTM 43N.

In [ ]:
df=pd.read_csv('Train_Test_REE_final.csv')
df.dropna(inplace=True)
df

In [ ]:
# Inverse projection to get Long, Lat from UTM coordinates. Then we define WGS 1984(also called EPSG:4326) coordinate reference system(crs)
long,lat=myProj(df.X.values,df.Y.values, inverse=True)
geometry = [Point(xy) for xy in zip(long, lat)]                  # All locations converted to point geometry
df = GeoDataFrame(df, crs="EPSG:4326", geometry=geometry)        # Dataframe converted to Geospatial compatible dataframe(Geodataframe)
df

# ANN Starts here

In [ ]:
# Split the dataframe into training (80%) and testing (20%) sets randomly
df_tr, df_ts = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
X_tr = df_tr.iloc[:,3:-1].values
np.shape(X_tr)
X_tr = StandardScaler().fit_transform(X_tr)

In [ ]:
y_tr = df_tr['Prob'].values

In [ ]:
y_tr

In [ ]:
X_ts = df_ts.iloc[:,3:-1].values
np.shape(X_ts)
X_ts = StandardScaler().fit_transform(X_ts)
y_ts = df_ts['Prob'].values

## Define the ANN model and choosing best model parameters

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score, make_scorer

seed_value = 42
# Define the function to create the ANN model
def create_model(hidden_layers=1, neurons=32, dropout_rate=0.2, learning_rate=0.001):
    model = Sequential()
    
    # Input layer
    model.add(Dense(neurons, activation='relu', input_dim=X_tr.shape[1]))
    
    # Hidden layers
    for _ in range(hidden_layers):
        model.add(Dense(neurons, activation='relu'))
        model.add(Dropout(dropout_rate))
    
    # Output layer (regression task)
    model.add(Dense(1, activation='linear'))
    
    # Compile the model with Adam optimizer and MSE loss
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mean_squared_error')
    return model

# Wrap the Keras model for RandomizedSearchCV
model = KerasRegressor(model=create_model, verbose=0)

# Define the parameter grid for tuning
param_grid = {
    'model__hidden_layers': [1, 2, 3, 4],  # Number of hidden layers
    'model__neurons': [32, 64, 128],       # Number of neurons per layer
    'model__dropout_rate': [0.2, 0.3],     # Dropout rate
    'model__learning_rate': [0.001, 0.01], # Learning rate
    'batch_size': [16, 32],                # Batch size
    'epochs': [50, 100]                    # Number of epochs
}

# Custom scoring metrics
scorers = {
    'MSE': 'neg_mean_squared_error',
    'MAE': make_scorer(mean_absolute_error, greater_is_better=False),
    'R2': 'r2',
    'Explained_Variance': make_scorer(explained_variance_score)
}

# Add early stopping to avoid overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Perform RandomizedSearchCV, optimizing for R2
start_time = time.time()
random_search = RandomizedSearchCV(
    estimator=model, 
    param_distributions=param_grid, 
    n_iter=50,  # Limit to 50 parameter combinations
    cv=3, 
    scoring=scorers,  # Use multiple scorers
    refit='R2',  # Optimize for R²
    verbose=1, 
    random_state=seed_value,  # Ensure reproducibility
    n_jobs=1  # Prevent parallel execution randomness
)
random_search_result = random_search.fit(X_tr, y_tr, callbacks=[early_stopping])
end_time = time.time()

# Print time taken for RandomizedSearchCV
print(f"Time taken for RandomizedSearchCV: {end_time - start_time:.2f} seconds")

# Store results in a list
results = []
for i, params in enumerate(random_search_result.cv_results_['params']):
    y_pred_train = random_search_result.best_estimator_.model_.predict(X_tr)
    y_pred_test = random_search_result.best_estimator_.model_.predict(X_ts)

    # Compute Evaluation Metrics
    mse_train = mean_squared_error(y_tr, y_pred_train)
    mse_test = mean_squared_error(y_ts, y_pred_test)
    mae_train = mean_absolute_error(y_tr, y_pred_train)
    mae_test = mean_absolute_error(y_ts, y_pred_test)
    r2_train = r2_score(y_tr, y_pred_train)
    r2_test = r2_score(y_ts, y_pred_test)
    ev_train = explained_variance_score(y_tr, y_pred_train)
    ev_test = explained_variance_score(y_ts, y_pred_test)

    results.append({
        **params,
        'Train MSE': mse_train, 'Test MSE': mse_test,
        'Train MAE': mae_train, 'Test MAE': mae_test,
        'Train R2': r2_train, 'Test R2': r2_test,
        'Train Explained Variance': ev_train, 'Test Explained Variance': ev_test
    })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)
print("RandomizedSearchCV Results:")
print(results_df)

# Save results to a CSV file
results_df.to_csv("random_search_results.csv", index=False)



# Plot RandomizedSearchCV results (R²)
plt.figure(figsize=(10, 6))
r2_scores = random_search_result.cv_results_['mean_test_R2']
params = range(len(r2_scores))
plt.plot(params, r2_scores, marker='o', linestyle='-', label='R² Scores')
plt.title('RandomizedSearchCV Results (Optimizing R²)')
plt.xlabel('Parameter Combination Index')
plt.ylabel('Mean R² Score')
plt.legend()
plt.grid(True)
plt.show()

# Print best parameters and scores
print("Best Parameters:", random_search_result.best_params_)
print("Best R² Score:", random_search_result.best_score_)

# Train the best model on the full dataset
start_time = time.time()
best_model = random_search_result.best_estimator_
history = best_model.fit(X_tr, y_tr, validation_data=(X_ts, y_ts), verbose=1, callbacks=[early_stopping])
end_time = time.time()
print(f"Time taken to train the best model: {end_time - start_time:.2f} seconds")

# Extract training and validation losses
history_dict = history.history_

# Plot training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(history_dict['loss'], label='Training Loss')
plt.plot(history_dict['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Evaluate the best model on the test set
y_pred_train_best = best_model.model_.predict(X_tr)
y_pred_test_best = best_model.model_.predict(X_ts)

# Compute Evaluation Metrics for the Best Model
mse_train_best = mean_squared_error(y_tr, y_pred_train_best)
mse_test_best = mean_squared_error(y_ts, y_pred_test_best)
mae_train_best = mean_absolute_error(y_tr, y_pred_train_best)
mae_test_best = mean_absolute_error(y_ts, y_pred_test_best)
r2_train_best = r2_score(y_tr, y_pred_train_best)
r2_test_best = r2_score(y_ts, y_pred_test_best)
ev_train_best = explained_variance_score(y_tr, y_pred_train_best)
ev_test_best = explained_variance_score(y_ts, y_pred_test_best)

print(f"Best Model Performance Metrics:")
print(f"Train MSE: {mse_train_best:.4f}, Test MSE: {mse_test_best:.4f}")
print(f"Train MAE: {mae_train_best:.4f}, Test MAE: {mae_test_best:.4f}")
print(f"Train R² Score: {r2_train_best:.4f}, Test R² Score: {r2_test_best:.4f}")
print(f"Train Explained Variance: {ev_train_best:.4f}, Test Explained Variance: {ev_test_best:.4f}")

In [ ]:
# Model performance
print('Training data(Known to machine, human)')
mse = mean_squared_error(y_tr, y_train_pred)
rmse = mse**0.5
print(f"Training MSE: {mean_squared_error(y_tr, y_train_pred)}")
print(f"Training RMSE: {rmse}")
print(f"Training MAE: {mean_absolute_error(y_tr, y_train_pred)}")
print(f"Training R2 Score: {r2_score(y_tr, y_train_pred)}")
print(f"Training Explained Variance: {explained_variance_score(y_tr, y_train_pred)}")


print('Test data(Unknown to machine, known to human)')
mse = mean_squared_error(y_ts, y_test_pred)
rmse = mse**0.5
print(f"Test MSE: {mean_squared_error(y_ts, y_test_pred)}")
print(f"Training RMSE: {rmse}")
print(f"Test MAE: {mean_absolute_error(y_ts, y_test_pred)}")
print(f"Test R2 Score: {r2_score(y_ts, y_test_pred)}")
print(f"Test Explained Variance: {explained_variance_score(y_ts, y_test_pred)}")

In [ ]:
X = df.iloc[:,3:-1].values
np.shape(X)
X = StandardScaler().fit_transform(X)
X

In [ ]:
#X=df.iloc[:,2:-2].values
all_pred = final_model.model_.predict(X) 


In [ ]:
y=df['Prob'].values
print('Train+Test data Mean squared error=', mean_squared_error(y, all_pred))
df['Predicted_Probability']=all_pred

In [ ]:
df

In [ ]:
mdf=pd.read_csv('Master_REE.csv')
mdf.dropna(inplace=True)
mdf

In [ ]:
MX=mdf.iloc[:,2:].values
MX

In [ ]:
MX=mdf.iloc[:,2:].values
MX_scale = StandardScaler().fit_transform(MX)
all_pred = final_model.model_.predict(MX_scale) 
mdf['Predicted_Probability']=all_pred

In [ ]:
mdf

In [ ]:
mdf.to_csv('REE_ANN.csv', index=False)